[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C44_Adversarial_Security_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与威胁建模热身

本课全程 **纯 numpy、CPU 可跑**，用**玩具模型 + 合成数据**复现攻击，并**立刻配上检测/防御**。

> **立场与边界**：讲攻击是为了防御。所有演示均为**小规模、教学性**，跑在我们完全掌握真相的玩具数据上，**不构成可用于危害真实系统的武器**。只在你自己拥有或获明确授权的系统上做安全实验。

这个 notebook 做四件事：① 确认环境；② 搭一个全课复用的**玩具分类器**；③ 用一张表把**攻击面（阶段×目标×能力）**结构化；④ 立下全课的纪律——**攻击 → 检测 → 防御** 三件套与「用数字说话」。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于可视化）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅  —— 本课无需 GPU / 联网 / 重型框架')

## 2 · 全课复用的玩具分类器

我们需要一个**自己完全掌握的**靶子。用合成的二维两类高斯数据训练一个 **logistic 回归**（纯 numpy、梯度下降）。

二维便于可视化、可解析；logistic 回归可微（方便算对抗样本梯度）、会过拟合（方便演示成员推断）、可被投毒——正好覆盖后面所有攻击。**我们知道它的全部参数和数据来源**，这正是教学所需。

In [ ]:
def make_blobs(n=400, d=2, sep=2.2, seed=0):
    '''两类各向同性高斯，均值相隔 sep。返回 X:(n,d), y:(n,)∈{0,1}。'''
    rng = np.random.default_rng(seed)
    n0 = n // 2; n1 = n - n0
    mu = np.zeros(d); mu[0] = sep
    X0 = rng.standard_normal((n0, d)) - mu / 2
    X1 = rng.standard_normal((n1, d)) + mu / 2
    X = np.vstack([X0, X1]); y = np.concatenate([np.zeros(n0), np.ones(n1)])
    perm = rng.permutation(n)
    return X[perm], y[perm].astype(int)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

class LogisticReg:
    '''最简 logistic 回归：w,b 梯度下降；可算对输入的梯度（对抗样本要用）。'''
    def __init__(self, d):
        self.w = np.zeros(d); self.b = 0.0
    def logits(self, X):
        return X @ self.w + self.b
    def prob(self, X):
        return sigmoid(self.logits(X))
    def predict(self, X):
        return (self.prob(X) > 0.5).astype(int)
    def fit(self, X, y, lr=0.1, epochs=300, l2=1e-3):
        n = len(y)
        for _ in range(epochs):
            p = self.prob(X)
            gw = X.T @ (p - y) / n + l2 * self.w
            gb = np.mean(p - y)
            self.w -= lr * gw; self.b -= lr * gb
        return self

def accuracy(model, X, y):
    return float(np.mean(model.predict(X) == y))

X, y = make_blobs()
Xtr, ytr, Xte, yte = X[:300], y[:300], X[300:], y[300:]
clf = LogisticReg(d=2).fit(Xtr, ytr)
acc = accuracy(clf, Xte, yte)
print(f'玩具分类器测试精度 = {acc:.3f}')
assert acc > 0.82, '可分数据上应当轻松 >0.8'
print('✅ 靶子就绪：一个我们完全掌握参数与数据的 logistic 分类器')

## 3 · 把攻击面结构化：阶段 × 目标 × 能力

安全分析的第一步永远是**威胁建模**。用一张表把本课覆盖的攻击按 NIST AML 的三主轴归位——**生命周期阶段**、破坏的 **CIA 目标**、所需的**攻击者能力**。

这张表就是后面五个模块的地图：看懂「一个攻击属于哪一格」，就知道它的目标、可行性与该用哪类防御。

In [ ]:
# (攻击, 阶段, CIA目标, 攻击者能力, 对应模块)
ATTACKS = [
    ('对抗样本 FGSM/PGD',   'inference',  'Integrity',     'white/black-box 构造输入', '01'),
    ('数据投毒(可用性)',    'training',   'Availability',  '注入/篡改训练数据',        '02'),
    ('后门/trigger',        'training',   'Integrity',     '低比例投毒训练数据',       '02'),
    ('模型窃取',            'extraction', 'Confidentiality','查询 API',               '03'),
    ('模型反演',            'extraction', 'Confidentiality','查询 API + 置信度',       '03'),
    ('成员推断 MIA',        'extraction', 'Confidentiality','查询 API',               '04'),
    ('prompt 注入(间接)',   'deployment', 'Integrity/Misuse','影响模型读到的内容',     '05'),
    ('供应链投毒',          'deployment', 'Integrity',     '污染依赖/权重/数据',       '05'),
]
print(f"{'攻击':<20}{'阶段':<12}{'CIA目标':<18}{'能力':<22}{'模块'}")
for name, stage, goal, cap, mod in ATTACKS:
    print(f'{name:<20}{stage:<12}{goal:<18}{cap:<22}{mod}')

stages = {a[1] for a in ATTACKS}
assert stages == {'training', 'inference', 'extraction', 'deployment'}, '四个阶段都应覆盖'
print('\n✅ 攻击面覆盖训练/推理/提取/部署四阶段 —— 这就是全课地图')

## 4 · 防御视角：攻击 → 检测 → 防御 三件套

本课每个攻击都遵循同一条纪律。先用一个**最小例子**走一遍：

**攻击**：把一个测试样本推过决策边界让它被分错（对抗样本的雏形）。
**检测**：用「离决策边界异常近 / 置信度异常」当作可疑信号。
**防御**：对低置信预测**拒答（abstain）**，把「被骗」变成「我不确定」。

In [ ]:
# 取一个被正确分类、且较靠近边界的测试点
probs = clf.prob(Xte)
correct = clf.predict(Xte) == yte
margin = np.abs(probs - 0.5)               # 离边界的距离（越小越不自信）
cand = np.where(correct)[0]
i = cand[np.argmin(margin[cand])]          # 最靠近边界的正确样本
x0, y0 = Xte[i].copy(), yte[i]
print(f'原始点: 预测={clf.predict(x0[None])[0]} 真标签={y0} 置信={clf.prob(x0[None])[0]:.3f}')

# 攻击：沿 w 方向推过边界（玩具版对抗扰动，模块01会正式做 FGSM/PGD）
step = 0.1; x_adv = x0.copy()
direction = -np.sign(clf.w) if y0 == 1 else np.sign(clf.w)
for _ in range(100):
    if clf.predict(x_adv[None])[0] != y0:
        break
    x_adv = x_adv + step * (clf.w if y0 == 0 else -clf.w)
p_adv = clf.prob(x_adv[None])[0]
print(f'对抗点: 预测={clf.predict(x_adv[None])[0]} (已翻转) 置信={p_adv:.3f}')
assert clf.predict(x_adv[None])[0] != y0, '扰动后应被分错'

# 防御：拒答机制 —— 置信度低于阈值就 abstain，不被迫给出（错误）答案
def predict_with_abstain(model, X, tau=0.15):
    p = model.prob(X); conf = np.abs(p - 0.5)
    pred = (p > 0.5).astype(int)
    pred[conf < tau] = -1                  # -1 表示拒答
    return pred

out = predict_with_abstain(clf, x_adv[None], tau=0.15)[0]
print(f'带拒答的防御: 输出={out} (-1=拒答)')
assert out == -1, '靠近边界的对抗点应触发拒答而非给出错误标签'
print('\n✅ 三件套跑通：攻击让它分错 → 检测到低置信 → 防御改为拒答（把“被骗”降级为“我不确定”）')

## 5 · 用数字说话：安全主张必须可度量

「这个系统安全吗」无法回答；「**在攻击者只能黑盒查询、扰动预算 ε=0.5 时，鲁棒精度是多少**」才能回答。

把全课会用到的核心指标列清楚——每个攻击/防御都要落到一个数字上，而不是「感觉安全」。

In [ ]:
METRICS = {
    '对抗样本': 'robust accuracy @ ε  （最坏扰动下的准确率，必与 clean acc 并列）',
    '后门':     'ASR 攻击成功率 + 干净精度  （带 trigger 被分到目标类的比例）',
    '窃取':     'fidelity @ query budget  （替身与目标预测的一致率 vs 查询数）',
    '成员推断': 'TPR @ low FPR  （低误报下能否精准指认成员，看 log-ROC 左端）',
    'prompt注入':'攻击成功率 + 防御后残余率  （注入指令被执行的比例）',
}
for k, v in METRICS.items():
    print(f'  {k:<10} → {v}')
assert len(METRICS) == 5
print('\n核心原则：没有数字的安全主张是空话。攻防是一场可度量的成本竞赛。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：① 玩具模型 + 合成数据，我们掌握全部真相；② 每个攻击都从零用 numpy 实现并打印「它确实奏效」的证据；③ 每个攻击都紧跟检测器与防御，并用 `assert` 验证防御有效；④ 用正确指标量化，绝不靠「感觉」。

**接下来五个模块**：01 对抗样本 → 02 投毒与后门 → 03 窃取与反演 → 04 成员推断与隐私 → 05 prompt 注入与供应链。

下一站：**模块 01 · 对抗样本（FGSM、PGD 与对抗训练）**。